# Setup

## Libraries 

In [ ]:
from pathlib import Path
import numpy as np
import cv2
print (cv2.__version__)
import os as os
import re
from utils import create_compound_image
import plotly.graph_objects as go
from collections import Counter
import matplotlib.pyplot as plt
import random
import plotly.subplots as sp

## Loading the images 

In [ ]:
files = os.listdir("images_and_poses_for_project_assignment")

print("Total files:", len(files))

ext = Counter([f.split(".")[-1] for f in files if "." in f])
print("Type:")
print(ext)

In [ ]:
BASE_PATH = os.getcwd()
IMAGE_PATH = os.path.join(BASE_PATH, "images_and_poses_for_project_assignment")

folderpath = IMAGE_PATH



images_path = [
    os.path.join(IMAGE_PATH, f)
    for f in os.listdir(IMAGE_PATH)
    if f.lower().endswith(".png")
]

# yaml
yaml_path = [
    os.path.join(IMAGE_PATH, f)
    for f in os.listdir(IMAGE_PATH)
    if f.lower().endswith(".yaml")
]

# extract the number form the filename
def extract_number(path):
    filename = os.path.basename(path)   #take only rgb_X.png
    return int(re.search(r'\d+', filename).group())

images_path.sort(key=extract_number)
yaml_path.sort(key=extract_number)

print(images_path[:5])
print(yaml_path[:5])

In [ ]:
shapes = []

for p in images_path:
    image = cv2.imdecode(np.fromfile(p, dtype=np.uint8), cv2.IMREAD_COLOR)
    shapes.append(image.shape)

print(set(shapes))

In [ ]:
import plotly.express as px

image = cv2.imdecode(
    np.fromfile(images_path[0], dtype=np.uint8),
    cv2.IMREAD_COLOR
)

In [ ]:
fig1 = px.imshow(image,width=900, height=600)
fig1.update_layout(title="BGR")
fig1.show()

In [ ]:
fig = px.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB),width=900, height=600)
fig.update_layout(title="RGB")
fig.show()


In [ ]:
rgb_images = []

for p in images_path:
    image = cv2.imdecode(np.fromfile(p, dtype=np.uint8), cv2.IMREAD_COLOR)
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    rgb_images.append(rgb)

print("Number of RGB images:", len(rgb_images))
print("Shape of first image:", rgb_images[0].shape)

In [ ]:
### Define checkerboard pattern size
grid_size = (8, 11)   #number of internal corners, choosing by counting how many corner interactions exist along the 2 directions of the checkerboard 
square_size = 11      #physical size in mm (for the real-world coordinates)


### Detect chessboard corners
Each image is converted to grayscale because corner detection is based on intensity gradients.
$cv2.findChessboardCorners$ detects the internal checkerboard intersections.
Only images where detection succeeds are stored for calibration.

In [ ]:
 
all_corners = []
valid_images = []
drawn_images=[]

for i, img in enumerate(rgb_images):
    
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) #convert to grayscale
    
    ret, corners = cv2.findChessboardCorners(gray, grid_size) #detect corners 
    
    if ret: #store the detected 2D points 
        all_corners.append(corners) 
        valid_images.append(i)
        drawn = cv2.drawChessboardCorners(
            img.copy(), grid_size, corners, ret
        )
        
        drawn_images.append(drawn)
    else:
        print(f"Pattern not found in image {i}")

print("Number of valid images:", len(all_corners))


In [ ]:
compound_image = create_compound_image(
    rows=4,
    cols=5,
    limages=drawn_images
)

fig = px.imshow(compound_image, width=900, height=600)
fig.update_layout(title="Detected chessboard corners")
fig.show()

# Zhang's method

Zhang's camera is a calibration method to estimate the camera intrinsics parameters $K$ and, for each calibration image, the extrinsic $R, t$ using a planar calibration pattern(checkerboard). 

The key idea is that for a planar pattern, the mapping between points on the plane and their image projections is a homography $H$. 

By estimating one homography per image and enforcing geometric constraints derived from the camera model, Zhang’s method provides a closed-form solution for the intrinsic parameters, followed by the extrinsic parameters for each view.

The camera model is:
$$
s\,x = P X
$$

where: 
$$
P = K [\,R \mid t\,]
$$

with:
$$
K = \text{intrinsic matrix}, \qquad R,t = \text{rotation and translation}, \qquad X = (X, Y, Z, 1)^{\top}, \qquad x = (u, v, 1)^{\top}
$$

Since the calibration object is planar, we have:

$$
s
\begin{bmatrix}
u \\
v \\
1
\end{bmatrix}
= K
\begin{bmatrix}
r_1 & r_2 & t
\end{bmatrix}
\begin{bmatrix}
X \\
Y \\
1
\end{bmatrix}
$$

Equal to:
$$
s\,x = H X_p
$$
So, for each image, the mapping between planar world points and image points is a homography $H$:
$$
H = K
\begin{bmatrix}
r_1 & r_2 & t
\end{bmatrix}, \qquad X_p = (X, Y, 1)^{\top}
$$

## 1. Build real-world coordinates 

We create a grid of 3D points corresponding to the checkerboard corners, the correspondences.

We construct the planar world coordinates ($X_i, Y_i$), associated to each detected image corner ($u_i, v_i$), scaled by the physical square size:

$$
X_i = u_{\text{index}}\cdot \text{square\_size}, \qquad
Y_i = v_{\text{index}}\cdot \text{square\_size}
$$

Since the checkerboard (calibration object) lies on a plane, all $Z_i$ are set to 0:
$$
X_i = (X_i, Y_i, 0, 1)^{\top}
$$


In [ ]:
#visualization step on a single image to verify the correctness 
k = 0
corners = all_corners[k].reshape(-1, 2)  # (N,2) = (u,v) in pixels

# load the corresponding image (RGB) just for visualization
idx = valid_images[k]  # index into images_path
image_bgr = cv2.imdecode(np.fromfile(images_path[idx], dtype=np.uint8), cv2.IMREAD_COLOR)
image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

another_copy = image.copy()

# real coords for this image: (N,2) in mm
real_coordinates = np.zeros((corners.shape[0], 2), dtype=np.float64)

grid_size_cv2 = tuple(reversed(grid_size))  # keep your exact alignment

for index, corner in enumerate(corners):
    u_coord, v_coord = corner[0], corner[1]  #each corner is a detected image point in pixel (u,v)

    # convert 1D index into 2D index (u_index, v_index) on the checkerboard grid
    u_index, v_index = np.unravel_index(index, grid_size_cv2)

    # convert grid indices to metric coordinates (mm): real world coordinates 
    x_mm = u_index * square_size
    y_mm = v_index * square_size

    real_coordinates[index, :] = [x_mm, y_mm]

    # draw label on the image
    cv2.putText(
        another_copy,
        text=f"{x_mm};{y_mm}",
        org=(int(round(u_coord)), int(round(v_coord))),
        fontFace=cv2.FONT_HERSHEY_SIMPLEX,
        fontScale=0.4,
        color=(255, 0, 0),
        thickness=1
    )

fig = px.imshow(another_copy, width=900, height=600)
fig.update_layout(title="Real-world (mm) coordinates overlaid on detected corners")
fig.show()

print("real_coordinates shape:", real_coordinates.shape)  # (N,2)
print("first 5 real coords (mm):\n", real_coordinates[:5])
print("first 5 image coords (px):\n", corners[:5])

In [ ]:
# create the set of correspondences for the entire dataset
grid_size_cv2 = tuple(reversed(grid_size))

img_points = []
obj_points = []

# Precompute the real-world coordinates once (same grid for all images)
N = grid_size[0] * grid_size[1] #total number of corners
obj_template = np.zeros((N, 2), dtype=np.float64)

for index in range(N):
    u_index, v_index = np.unravel_index(index, grid_size_cv2) #convert 1D index into 2D index (u_index, v_index)
    obj_template[index, 0] = u_index * square_size
    obj_template[index, 1] = v_index * square_size

# Create correspondences for each image
for corners_k in all_corners:
    corners_uv = corners_k.reshape(-1, 2).astype(np.float64)  #(N,2)
    img_points.append(corners_uv) #imagines points in pixel (u,v)
    obj_points.append(obj_template.copy()) #real-world points in mm (x,y)

print("Images with correspondences:", len(img_points)) #81 images 
print("Points per image:", img_points[0].shape[0]) #8x11=88


## 2. Estimating the homography $H$
Using the planar correspondences ($X_i, Y_i$), we estimate a homography $H$ for each image using the Direct Linear Transform (DLT), without knowing $K$:
$$
H^{(k)} = K
\begin{bmatrix}
r_1^{(k)} & r_2^{(k)} & t^{(k)}
\end{bmatrix}
$$
 For each correspondence, two linear equations are added to matrix $A$, resulting in a homogeneous system:
 $$Ah=0 , \qquad A \in \mathbb{R}^{2N \times 9} $$

Since the is effected by noise, the homography is obtained as the singular vector associated with the smallest singular value of $A$, so using SVD:
  $$ A = U \Sigma V^{\top} $$
Since the homography is defined up to scale , we normalized:
$$
H \sim \lambda H, \qquad H_{33} = 1
$$





In [ ]:
def compute_homography(obj_pts, img_pts):
    """
    obj_pts: (N,2) planar world coordinates (X,Y)
    img_pts: (N,2) image coordinates (u,v)
    returns H (3x3)
    """

    N = obj_pts.shape[0]
    A = np.zeros((2*N, 9))

    for i in range(N):
        X, Y = obj_pts[i]
        u, v = img_pts[i]

        A[2*i] =   [X, Y, 1, 0, 0, 0, -u*X, -u*Y, -u] #first row of A for point i (equation for u)
        A[2*i+1] = [0, 0, 0, X, Y, 1, -v*X, -v*Y, -v] #second row of A  (equation for v)

    # Solve Ah = 0 using the SVD
    U, S, Vt = np.linalg.svd(A)

    h = Vt[-1]        # last singular vector, (9,1)
    H = h.reshape(3,3) #convert into a 3x3 matrix

    # normalize so that H[2,2] = 1
    H = H / H[2,2]

    return H


In [ ]:
H_list = [] #store the homography for each image

for k in range(len(img_points)):
    H_k = compute_homography(obj_points[k], img_points[k])
    H_list.append(H_k)

print("Number of homographies:", len(H_list))
print("Example H:\n", H_list[0])


## 3. Estimating the intrinsic matrix K from H
Now, since we have estimated the homography for each calibration $k$, we can estimate the intrinsic matrix $K$.
The idea is that $K$ can be written as linear constraints on a symmetric matrix $B$: 
$$ B = (K K^{\top})^{-1} $$

Because $r_1$ and $r_2$ are orthonormal:
$$
r_1^{\top} r_2 = 0 \;\;\Rightarrow\;\; h_1^{\top} B h_2 = 0
$$

$$
r_1^{\top} r_1 = r_2^{\top} r_2 \;\;\Rightarrow\;\; h_1^{\top} B h_1 = h_2^{\top} B h_2
$$

Each constraint can be written in linear form:
$$
h_i^{\top} B h_j = v_{ij}^{\top} b , \qquad v_{ij} =
\begin{bmatrix}
h_{1i} h_{1j} \\
h_{1i} h_{2j} + h_{2i} h_{1j} \\
h_{2i} h_{2j} \\
h_{3i} h_{1j} + h_{1i} h_{3j} \\
h_{3i} h_{2j} + h_{2i} h_{3j} \\
h_{3i} h_{3j}
\end{bmatrix}
$$

So, each homography provides 2 linear equations: 
$$
v_{12}^{\top} b = 0
$$

$$
\left( v_{11} - v_{22} \right)^{\top} b = 0
$$

We obtain a homogeneous linear system, solved via SVD: 
$$
V b = 0, \qquad 
V \in \mathbb{R}^{2M \times 6}


$$

The solution $b$ is the right singular vector corresponding to the smallest singular value of $V$. 
Finally, via Cholesky factorization, we recover $K$ from $B = (K K^{\top})^{-1}$.

In [ ]:
def v_ij(H, i, j):
    """
    Find the v_ij vector from Zhang's method 
    
    """
    return np.array([
        H[0,i]*H[0,j],
        H[0,i]*H[1,j] + H[1,i]*H[0,j],
        H[1,i]*H[1,j],
        H[2,i]*H[0,j] + H[0,i]*H[2,j],
        H[2,i]*H[1,j] + H[1,i]*H[2,j],
        H[2,i]*H[2,j]
    ], dtype=np.float64)

#Build V from all homographies
V_rows = []
for H in H_list:
    # v12^T b = 0
    V_rows.append(v_ij(H, 0, 1))
    # (v11 - v22)^T b = 0
    V_rows.append(v_ij(H, 0, 0) - v_ij(H, 1, 1))

V = np.stack(V_rows, axis=0)  #  (2*M, 6)
print("V shape:", V.shape)

#Solve V b = 0 via SVD 
_, _, Vt = np.linalg.svd(V)
b = Vt[-1, :]  #smallest singular value solution
print("b:", b)

#Reconstruct symmetric B from b
B = np.array([
    [b[0], b[1], b[3]],
    [b[1], b[2], b[4]],
    [b[3], b[4], b[5]]
], dtype=np.float64)
print("B:\n", B)

#Recover K from B  
if np.linalg.det(B) < 0: #B is defined up to scale; if sign is flipped, fix it
    B = -B

#Cholesky: B = L L^T  (requires B to be positive definite)
L = np.linalg.cholesky(B)

#K^{-1} = L^T  => K = (L^T)^{-1}
K = np.linalg.inv(L.T)

# Normalize K so that K[2,2] = 1 (fix scale)
K = K / K[2,2]

print("Estimated intrinsic matrix K:\n", K)


## 4. Estimating the extrinsic ($R_k, t_k$) from $H_k$ and $K$

Now, for each image $k$, we can estimate the extrinsic parameters from: 
$$
H^{(k)} \sim K
\begin{bmatrix}
r_1^{(k)} & r_2^{(k)} & t^{(k)}
\end{bmatrix}
$$
So: 
$$
K^{-1} h_1^{(k)} = \lambda_k\, r_1^{(k)}, \qquad
K^{-1} h_2^{(k)} = \lambda_k\, r_2^{(k)}, \qquad
K^{-1} h_3^{(k)} = \lambda_k\, t^{(k)}, \qquad \lambda_k = \frac{1}{\left\| K^{-1} h_1^{(k)} \right\|} \qquad (\text{same for } h_2^{(k)})
$$
Then for orthogonality: 
$$
r_3^{(k)} = r_1^{(k)} \times r_2^{(k)}
$$

Do to noise, the matrix R can not be orthogonal, so we take the closest orthogonal matrix and the singular value decomposition of $R$:
$$
R^{(k)} = U V^{\top}
\qquad\text{where}\qquad
R_{\text{raw}}^{(k)} = U \Sigma V^{\top}
$$

Finally, we obtain the extrinsic parameters for each image $R_k, t_k$.

In [ ]:
def extrinsics_from_H(K, H):
    """
    Input:
      K: (3,3) intrinsic matrix
      H: (3,3) homography of one image
    Output:
      R: (3,3) rotation matrix
      t: (3,)  translation vector
    """
    Kinv = np.linalg.inv(K) #inverse of K

    #take the columns of H
    h1 = H[:, 0]
    h2 = H[:, 1]
    h3 = H[:, 2]

    #scale factor lambda
    lam = 1.0 / np.linalg.norm(Kinv @ h1)
    
    #rotation columns 
    r1 = lam * (Kinv @ h1)
    r2 = lam * (Kinv @ h2)
    r3 = np.cross(r1, r2)
    
    #raw rotation 
    R_raw = np.column_stack((r1, r2, r3))
    
    #translation
    t = lam * (Kinv @ h3)

    #enforce R to be a proper rotation using SVD: 
    U, _, Vt = np.linalg.svd(R_raw)
    R = U @ Vt

    #the determinant of R must be positive
    if np.linalg.det(R) < 0:
        U[:, -1] *= -1
        R = U @ Vt

    return R, t

# Compute extrinsics for all images
Rt_list = []
for H in H_list:
    R, t = extrinsics_from_H(K, H)
    Rt_list.append((R, t))

print("Computed extrinsics:", len(Rt_list))
print("Example R:\n", Rt_list[0][0])
print("Example t:\n", Rt_list[0][1])
R0, t0 = Rt_list[0]
print("det(R0) =", np.linalg.det(R0)) #check determinant 
print("||R^T R - I||:", np.linalg.norm(R0.T @ R0 - np.eye(3))) #check the orthonormality of R


After finding the extrinsic parameters $R$ and $t$, it's important to verify their physical consistency. In particular, we check that the reconstructed chessboard points lie in the front of the camera, so if their Z-coordinate in the camera reference frame is positive. 

In [ ]:
def check_all_extrinsics(Rt_list, grid_size, square_size):
    bad_indices = []

    W = (grid_size[1]-1) * square_size #width of the checkerboard in mm
    H = (grid_size[0]-1) * square_size #height of the checkerboard in mm

    #4 corner coordinations of the chessboard (Z=0)
    Xw = np.array([
        [0, 0, 0], #top-left corner in world coordinates
        [W, 0, 0], #top-right corner in world coordinates
        [0, H, 0], #bottom-left corner in world coordinates
        [W, H, 0] #bottom-right corner in world coordinates
    ], dtype=float)

    for i, (R, t) in enumerate(Rt_list):
        Xc = (R @ Xw.T + t.reshape(3,1)).T #transformation of the 4 corners into camera coordinates 
        min_z = np.min(Xc[:, 2]) #take coordinate Z of the 4 points and find the minimum

        if min_z <= 0: #if the depth is negative or zero, it means the camera is not in front of the checkerboard plane
            bad_indices.append((i, min_z))

    return bad_indices

bad = check_all_extrinsics(Rt_list, grid_size, square_size)

if len(bad) == 0:
    print("All views are physically valid (all Z > 0).")
else:
    print("Problematic views found:")
    for idx, z in bad:
        print(f"Image {idx}: min Z = {z}")


# Total projection error

The geometric reprojection error is computed by projecting the planar grid points onto the image using the estimated parameters $K$,$R$,$t$, and measuring the Euclidean distance between the observed image points and the projected ones. 
Then, the total reprojection error is obtained by summing the individual distances over all grid points:
$$
\varepsilon(P)=
\sum_{i=1}^{n}
\left(
\frac{p_1^{\top} m^{(i)}}{p_3^{\top} m^{(i)}} - u^{(i)}
\right)^{2}
+
\left(
\frac{p_2^{\top} m^{(i)}}{p_3^{\top} m^{(i)}} - v^{(i)}
\right)^{2}

$$

$$
\begin{array}{rl}
\text{where:}& \\

& P = K[R \mid t] \text{ Projection matrix od the camera}\\

& p_1^{\top},\, p_2^{\top},\, p_3^{\top} \text{ rows of } P\\

& m^{(i)} \text{is the point 3D } i\text{-esimo in homogeneous coordinates}\\

& (u^{(i)}, v^{(i)}) \text{ correspondence point}
\end{array}
$$



In [ ]:
k = 0 #selection of the first image for reprojection error computation

R, t = Rt_list[k]
img_pts = img_points[k]
obj_pts = obj_points[k]

P = K @ np.column_stack((R, t.reshape(3,1))) #projection matrix

p1 = P[0, :] #rows of P
p2 = P[1, :]
p3 = P[2, :]

N = obj_pts.shape[0]
m = np.column_stack((obj_pts, np.zeros(N), np.ones(N))) #convert (X,Y) to homogeneous (X,Y,0,1)

proj_pts = []
epsilon = 0

for i in range(N):
    mi = m[i]

    u_proj = (p1 @ mi) / (p3 @ mi) #projected u coordinate in pixel
    v_proj = (p2 @ mi) / (p3 @ mi) #projected v coordinate in pixel

    proj_pts.append([u_proj, v_proj])

    u_obs = img_pts[i, 0]
    v_obs = img_pts[i, 1]

    epsilon += (u_proj - u_obs)**2 + (v_proj - v_obs)**2 #accumulate squared reprojection error for point i

proj_pts = np.array(proj_pts)

print("Total geometric reprojection error ε(P):", epsilon)
print("Mean reprojection error:", epsilon / N)


In [ ]:
#image RGB
image_bgr = cv2.imdecode(np.fromfile(images_path[k], dtype=np.uint8), cv2.IMREAD_COLOR)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

fig = go.Figure()


fig.add_trace(go.Image(z=image_rgb))

#observed points (green)
fig.add_trace(go.Scatter(
    x=img_pts[:,0],
    y=img_pts[:,1],
    mode='markers',
    name='Observed points',
    marker=dict(size=6)
    # # for project report image:
    # mode='markers',
    # name='Observed points',
    # marker=dict(size=10, symbol='x',color='green')
))

#reprojected points (red)
fig.add_trace(go.Scatter(
    x=proj_pts[:,0],
    y=proj_pts[:,1],
    mode='markers',
    name='Reprojected points',
    marker=dict(size=8, symbol='x')
    # # for project report image:
    # mode='markers',
    # name='Reprojected points',
    # marker=dict(size=6, color='red')
))


#for project report image: figure_width=1500, figure_height=1500
fig.update_yaxes(autorange="reversed")
fig.update_layout(
    width= 900,
    height=600,
    title="Observed vs Reprojected Points"
)

fig.show()


# Superimpose an object

We already have estimated:
- Intrinsic matrix $K$
- Extrinsic parameters $R,t$ for each image

This model allows us to project any 3D point expressed in the chessboard reference system onto the corresponding 2D image plane.

To qualitatively validate the calibration results, we superimpose a virtual 3D object (in our case, a cylinder) onto the calibration pattern across multiple images. The object is defined in the same 3D coordinate system as the chessboard, with its base lying on the plane $Z=0$.
Using the projection matrix: 

$$
P = K [\,R \mid t\,]
$$

we project the 3D model of the cylinder into each image and visually inspect the alignment.

If the calibration is correct:

- The base of the object remains attached to the chessboard plane

- The perspective distortion is geometrically consistent across different viewpoints

- The vertical edges of the cylinder follow the correct projective behavior

In [ ]:
def project_points(K, R, t, Xw):
    """
    K: (3,3)
    R: (3,3)
    t: (3,)
    Xw: (N,3) coordinations points (world)
    return: (N,2) pixel coordinates 
    """
    P = K @ np.column_stack((R, t.reshape(3,1)))     #projection matrix (3x4)
    Xh = np.column_stack((Xw, np.ones((Xw.shape[0], 1))))  #homogeneous coordinates (N,4)
    x = (P @ Xh.T).T                                  #projected points in homogeneous coordinates (N,3)
    x = x[:, :2] / x[:, 2:3]                          #division by z to get pixel coordinates
    return x

In [ ]:
def make_cylinder(center_xy, radius, height, n_circle=80, n_vertical=12):
    """
    center_xy: (cx, cy) base center on the plane z=0
    radius: r
    height: h
    n_circle: points to approximate the circles
    n_vertical: how many vertical lines to draw
    """
    cx, cy = center_xy
    angles = np.linspace(0, 2*np.pi, n_circle, endpoint=True) #angles for approximating the circles

    #base circle 
    base = np.column_stack([
        cx + radius*np.cos(angles),  #x
        cy + radius*np.sin(angles), #y
        np.zeros_like(angles) #z=0
    ])

    #top circle
    top = base.copy()
    top[:, 2] = height #set z coordinate to height

    #vertical lines: 
    v_angles = np.linspace(0, 2*np.pi, n_vertical, endpoint=False)
    verticals = []
    for a in v_angles:
        xb = cx + radius*np.cos(a)
        yb = cy + radius*np.sin(a)
        
         #create a line segment from (xb, yb, 0) to (xb, yb, height) to see the tridimensional structure
        verticals.append(( 
            np.array([[xb, yb, 0.0]]),
            np.array([[xb, yb, height]])
        ))

    return base, top, verticals


In [ ]:
def draw_polyline(img_bgr, pts2d, closed=True, thickness=6):
    pts = np.round(pts2d).astype(np.int32) #convert to int
    pts = pts.reshape(-1, 1, 2)
    #draw the upper and lower circles
    cv2.polylines(img_bgr, [pts], isClosed=closed, color=(0, 0, 255), thickness=thickness)  #red (BGR)

def draw_segment(img_bgr, p0, p1, thickness=6): 
    p0i = tuple(np.round(p0).astype(int)) #convert to int and tuple (pixel)
    p1i = tuple(np.round(p1).astype(int))
    #draw the vertical line
    cv2.line(img_bgr, p0i, p1i, color=(0, 255, 0), thickness=thickness)  # verde (BGR)



In [ ]:
max_x = (grid_size[1]-1) * square_size   #max x coordinate of the checkerboard in mm (7 squares * 11 mm = 77 mm)
max_y = (grid_size[0]-1) * square_size   #max y coordinate of the checkerboard in mm (10 squares * 11 mm = 110 mm)

center_xy = (0.5 * max_x, 0.5 * max_y)   #put the center of the cylinder in the middle of the checkerboard
radius = 2.0 * square_size               #radius: 2 squares
height = 6.0 * square_size               #height: 6 squares

base3d, top3d, verticals3d = make_cylinder(center_xy, radius, height, n_circle=100, n_vertical=12)


In [ ]:
out_dir = "overlay_cylinder"
os.makedirs(out_dir, exist_ok=True)

#take 25 images uniformly distributed across the dataset 
n_imgs = len(images_path)
idxs = np.linspace(0, n_imgs-1, 25).round().astype(int) #4 for report image

alpha = 0.35 

for k in idxs:
    R, t = Rt_list[k]

    #image
    image_bgr = cv2.imdecode(np.fromfile(images_path[k], dtype=np.uint8), cv2.IMREAD_COLOR)

    #project the circles and verticals
    base2d = project_points(K, R, t, base3d)
    top2d  = project_points(K, R, t, top3d)

    overlay = image_bgr.copy()

    base_pts = np.round(base2d).astype(np.int32)
    top_pts  = np.round(top2d).astype(np.int32)

    #fill top face
    cv2.fillPoly(overlay, [top_pts.reshape(-1, 1, 2)], color=(0, 0, 255))  # red (BGR)

    #fill lateral faces (quads)
    n = len(base_pts)
    for i in range(n - 1):
        quad = np.array([base_pts[i], base_pts[i+1], top_pts[i+1], top_pts[i]], dtype=np.int32)
        cv2.fillPoly(overlay, [quad.reshape(-1, 1, 2)], color=(0, 255, 0))  # green (BGR)

    #blend overlay with the original image
    image_bgr = cv2.addWeighted(overlay, alpha, image_bgr, 1 - alpha, 0)

    #draw the circles
    draw_polyline(image_bgr, base2d, closed=True, thickness=2)
    draw_polyline(image_bgr, top2d, closed=True, thickness=2)

    #draw vertical lines
    for p0_3d, p1_3d in verticals3d:
        p0_2d = project_points(K, R, t, p0_3d)[0]
        p1_2d = project_points(K, R, t, p1_3d)[0]
        draw_segment(image_bgr, p0_2d, p1_2d, thickness=2)

    #save
    out_path = os.path.join(out_dir, f"cylinder_overlay_{k:03d}.png")
    cv2.imwrite(out_path, image_bgr)

print(f"save {len(idxs)} images in: {out_dir}/")



In [ ]:
k_show = idxs[11] #choose an image among the 25
img_show = cv2.imread(os.path.join(out_dir, f"cylinder_overlay_{k_show:03d}.png"))
img_show_rgb = cv2.cvtColor(img_show, cv2.COLOR_BGR2RGB)

fig = go.Figure()
fig.add_trace(go.Image(z=img_show_rgb))
fig.update_layout(title=f"Cylinder overlay - image {k_show}", margin=dict(l=0,r=0,t=30,b=0))
fig.show()

In [ ]:
#plot of all the 25 images
rows = 5 #2 for report image
cols = 5 #2 for report image

fig = sp.make_subplots(rows=rows, cols=cols, horizontal_spacing=0.0, vertical_spacing=0.0)

for i, k_show in enumerate(idxs):
    row = i // cols + 1
    col = i % cols + 1

    img_show = cv2.imread(os.path.join(out_dir, f"cylinder_overlay_{k_show:03d}.png"))
    img_show_rgb = cv2.cvtColor(img_show, cv2.COLOR_BGR2RGB)

    fig.add_trace(go.Image(z=img_show_rgb), row=row, col=col)


fig.update_layout(
    height=1000,  
    width=1000,    
    title="Cylinder overlay on 25 images",
    margin=dict(l=15, r=15, t=40, b=15), #(l=0, r=0, t=0, b=0),  for report image
    showlegend=False
)


fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)

fig.show()


# Influence of the number of Calibration images on accuracy

In this experiment, we analyze how the number of calibration images affects the stability of the estimated intrinsic parameters, with particular focus on the principal coordinates $u_0$ and $v_0$.
Starting from the homographies previously computed for all calibration images, we perform repeated calibrations using only subsets of size $k$. For each subset, the intrinsic matrix $K$ is re-estimated using Zhang’s method. The principal point coordinates $u_0= K_{0,2}$ and $v_0= K_{1,2}$

For each value of $k$, multiple random subsets are generated, and the standard deviations $\sigma(u_0)$ and $\sigma(v_0)$ are computed. This procedure allows us to measure how sensitive the calibration result is to the choice and number of images used. A decreasing $\sigma(u_0)$ and $\sigma(v_0)$ indicates improved robustness and stability of the calibration matrix.

In [ ]:
def estimate_K_from_H_list(H_sub):
    """
    Re-estimate K using Zhang's method, using only the homographies in H_sub.
    """
    V_rows = []
    for H in H_sub:
        V_rows.append(v_ij(H, 0, 1))                 #v12
        V_rows.append(v_ij(H, 0, 0) - v_ij(H, 1, 1)) #v11 v22

    V = np.stack(V_rows, axis=0) #matrix V for the subset of homographies
    _, _, Vt = np.linalg.svd(V)
    b = Vt[-1, :] #solution of Vb=0, using SVD , last row of Vt

    #matrix B from vector b
    B = np.array([ 
        [b[0], b[1], b[3]],
        [b[1], b[2], b[4]],
        [b[3], b[4], b[5]]
    ], dtype=np.float64)

    #fix ambiguity of sign
    if np.linalg.det(B) < 0:
        B = -B

    #apply Cholesky to recover K 
    L = np.linalg.cholesky(B)
    K_est = np.linalg.inv(L.T)
    K_est = K_est / K_est[2, 2]
    return K_est

In [ ]:
def focal_from_K(K):
    #Use average of fx and fy as a single focal length measure 
    fx, fy = K[0, 0], K[1, 1]
    return 0.5 * (fx + fy)


In [ ]:
random.seed(0)
np.random.seed(0)
M = len(H_list) #total number of homographies available (images)
k_min = 2  #minimum number of homographies 
k_max = min(M, 18)      
n_trials = 500   #number of random subsets to sample for each k 

k_values = []
sigma_f_values = [] #focal 
sigma_u0_values = []
sigma_v0_values = []

#Monte carlo: casual subset of each size k 
for k in range(k_min, k_max + 1):
    f_estimates = []
    u0_estimates = []
    v0_estimates = []
    for _ in range(n_trials): #for each trial, take a random subset of k homographies
        idx = random.sample(range(M), k) #select k random indices 
        H_sub = [H_list[i] for i in idx] #subset of corresponding homographies

        try:
            K_est = estimate_K_from_H_list(H_sub) #estimate K from the subset of homographies
            f_estimates.append(focal_from_K(K_est)) #extract focal length
            u0_estimates.append(K_est[0,2])
            v0_estimates.append(K_est[1,2])
        except np.linalg.LinAlgError:
            #Cholesky may fail: skip
            pass
        
    #standard deviation calculation 
    if len(f_estimates) >= 2:
        k_values.append(k)
        sigma_f_values.append(np.std(f_estimates)) #how much changes f, using only k images
        sigma_u0_values.append(np.std(u0_estimates))
        sigma_v0_values.append(np.std(v0_estimates))



In [ ]:
plt.figure()
plt.plot(k_values, sigma_u0_values, marker='o', label='Std(u0)')
plt.plot(k_values, sigma_v0_values, marker='o', label='Std(v0)')
plt.xlabel("Number of images")
plt.ylabel("Std. deviation of principal point coordinates")
plt.title("Influence of the number of calibration images on (u0, v0)")
plt.legend()
plt.grid(True)
plt.show()

We can see that a very high variability is observed when only a small number of images is used, indicating that the estimation of the principal point is highly unstable in this regime. As the number of images increases, both $\sigma(u_0)$ and $\sigma(v_0)$ decrease rapidly and approach zero. From approximately six images onward, the estimation becomes stable and largely insensitive to the specific subset selection, confirming that a sufficient number of calibration views significantly improves the robustness of the intrinsic parameter estimation.

For completeness, the same analysis is also performed on the focal length $f$, in order to compare its stability with that of the principal point parameters.

In [ ]:
plt.figure()
plt.plot(k_values, sigma_f_values, marker='o')
plt.xlabel("Number of images")
plt.ylabel("Std. deviation of focal length f")
plt.title("Influence of the number of calibration images on accuracy")
plt.grid(True)
plt.show()

The standard deviation of the estimated focal length is very high when only a few images are used ($k=2$ and $k=3$) , indicating that the intrinsic estimation is unstable in this regime. As the number of images increases, the variability rapidly decreases, and from approximately $k=6$ onward the curve stabilizes close to zero, showing that the estimation becomes robust once sufficient geometric constraints are available.

Small local fluctuations may still appear for intermediate values of $k$. These variations are caused by the specific random subsets selected under a fixed seed, which determines the sequence of sampled subsets. Since the curve is based on a single random initialization, the result may partially depend on this Monte-Carlo sampling. To ensure that the observed behavior is not due to a particular random configuration, the experiment is repeated across multiple seeds and the results are averaged, providing a more reliable and statistically stable estimate of the relationship between the number of calibration images and parameter stability.

In [ ]:
def sigma_curve_for_seed(seed, H_list, k_min=2, k_max=18, n_trials=200):
    """
    Calculate the standard deviation of f for different values of k (as before), 
    using a specific random seed for reproducibility.
    """
    random.seed(seed)
    np.random.seed(seed)

    M = len(H_list)
    k_max = min(k_max, M)

    k_values = []
    sigma_values = []

    for k in range(k_min, k_max + 1):
        f_estimates = []

        for _ in range(n_trials):
            idx = random.sample(range(M), k)
            H_sub = [H_list[i] for i in idx]

            try:
                K_est = estimate_K_from_H_list(H_sub) #estimate of k
                f_estimates.append(focal_from_K(K_est)) #estimate of f
            except np.linalg.LinAlgError:
                pass

        if len(f_estimates) >= 2:
            k_values.append(k)
            sigma_values.append(np.std(f_estimates)) #standard deviation of f

    return np.array(k_values), np.array(sigma_values)


seeds = [0, 1, 2, 3, 4]   #different random seeds for multiple runs of the Monte Carlo simulation
n_trials = 500               
k_min = 2
k_max = 18

all_sigma = []
for s in seeds:
    k_vals, sig = sigma_curve_for_seed(s, H_list, k_min=k_min, k_max=k_max, n_trials=n_trials)
    all_sigma.append(sig)

all_sigma = np.vstack(all_sigma)
sigma_mean = all_sigma.mean(axis=0)  #mean of the curves using different seeds
sigma_std  = all_sigma.std(axis=0)   #sd of the curves using different seeds 

plt.figure()
plt.plot(k_vals, sigma_mean, marker='o')
plt.fill_between(k_vals, sigma_mean - sigma_std, sigma_mean + sigma_std, alpha=0.2)
plt.xlabel("Number of images")
plt.ylabel("Std. deviation of focal length f")
plt.title("Influence of the number of calibration images on accuracy")
plt.grid(True)
plt.show()


We can see the standard deviation of the estimated focal length as a function of the number of calibration images, averaged across multiple random seeds. Each curve corresponding to a different seed is combined by computing the mean standard deviation, while the shaded area represents the variability between seeds.

Compared to the single-seed experiment, this plot provides a more reliable and statistically stable estimate of the relationship between the number of images and calibration accuracy. Although the variability is still very large when only a few images are used, the overall decreasing trend is clearly consistent across different random initializations. The narrow shaded region for larger values of $k$ indicates that the result becomes independent of the specific random subset selection once a sufficient number of images is used.